# DT2470 chord-recognition workbench

An interactive entry point for the **current project**: local-song inference, the hierarchical CNN + HSMM and Gaussian-HMM pipelines, model training, dataset evaluation, and inspection of generated results. The algorithms remain in the project's tested Python modules; cells here configure and call those implementations rather than maintaining notebook-only copies.

Run the setup and configuration cells first, then execute the workflow sections you need. Inference defaults to the hierarchical pipeline. Training and dataset-wide evaluation are opt-in because they can take a long time and require datasets that are not included in this repository. The older Tiny AAM experiment remains in [`TinyAAM_BeatNet_Gaussian_HMM_Full_Pipeline.ipynb`](./TinyAAM_BeatNet_Gaussian_HMM_Full_Pipeline.ipynb).

For a conceptual overview, see [`ARCHITECTURE.md`](../ARCHITECTURE.md).

## 1. Environment and project setup

Use the project environment and install its declared dependencies before running the Python cells. From the repository root, run `python3 setup_env.py` in a terminal, then select that environment as this notebook's kernel. The core dependencies are listed in [`requirements.txt`](../requirements.txt); BeatNet is optional and has its own [`requirements-beatnet.txt`](../requirements-beatnet.txt).

The next cell locates the repository from the notebook's working directory and adds it to the Python import path.

In [20]:
from pathlib import Path
import os
import sys

candidates = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next((path for path in candidates if (path / "main.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root. Open this notebook from the repository, "
        "or change the notebook working directory to the project folder."
    )
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.executable}")

Project root: /Users/hanneswesterberg/Documents/GitHub/dt2470-project
Python: /Users/hanneswesterberg/Documents/GitHub/dt2470-project/.venv/bin/python


In [21]:
import csv
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, FileLink, display

from data.load_data import find_input_audio
from main import build_parser as build_inference_parser, run as run_inference

INPUT_DIR = PROJECT_ROOT / "input"
OUTPUT_DIR = PROJECT_ROOT / "output" / "notebook"
MODELS_DIR = PROJECT_ROOT / "models"
INPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Core project imports succeeded.")

Core project imports succeeded.


## 2. Select audio and configure inference

Place supported audio files (`.wav`, `.flac`, `.mp3`, or `.ogg`) in `input/`. For local inference, set `SONG_NAMES` to filenames or stems; an empty list runs all local files. To use the mounted AAM dataset, switch `SOURCE` to `cloud` and provide song IDs. RWC audio inference requires setting the `RWC_AUDIO_DIR` environment variable to the folder containing the RWC audio.

For the Gaussian baseline, select `PIPELINE = "gaussian"` and provide one or more complete model names/paths in `GAUSSIAN_MODELS`. The hierarchical model is the default and uses the artifacts under `models/hierarchical/`.

In [22]:
PIPELINE = "hierarchical"  # "hierarchical" or "gaussian"
SOURCE = "local"  # "local", "cloud", or "rwc"
SONG_NAMES = []  # e.g. ["example.wav"], or [] for all local input files
GAUSSIAN_MODELS = []  # e.g. ["gaussian_hmm_80"] when PIPELINE == "gaussian"
BEAT_TRACKER = "auto"  # Gaussian only: "auto", "librosa", or "beatnet"
DEVICE = None  # None chooses the model's default; otherwise e.g. "cpu", "mps", "cuda"
TOP_ROOTS = 4
HSMM_ORDER = 2
COMPARE_GROUND_TRUTH = True

# Set these only if the AAM audio/annotation directories are not discoverable
# from the project's Drive configuration.
AUDIO_DATASET_DIR = None
ANNOTATION_DATASET_DIR = None

local_audio_files = find_input_audio(INPUT_DIR)
print(f"Local audio files: {len(local_audio_files)}")
for audio_file in local_audio_files:
    print(f"  {audio_file.name}")
if not local_audio_files and SOURCE == "local":
    print(f"Add an audio file to {INPUT_DIR} before running local inference.")

Local audio files: 2
  10 - The Long And Winding Road (Remastered 2009).mp3
  HEY JUDE.mp3


In [23]:
if PIPELINE not in {"hierarchical", "gaussian"}:
    raise ValueError("PIPELINE must be 'hierarchical' or 'gaussian'.")
if SOURCE not in {"local", "cloud", "rwc"}:
    raise ValueError("SOURCE must be 'local', 'cloud', or 'rwc'.")
if PIPELINE == "gaussian" and not GAUSSIAN_MODELS:
    raise ValueError(
        "Set GAUSSIAN_MODELS to at least one trained Gaussian model name or path."
    )

cli_args = [
    "--pipeline", PIPELINE,
    "--input-dir", str(INPUT_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--models-dir", str(MODELS_DIR),
    "--source", SOURCE,
    "--non-interactive",
    "--top-roots", str(TOP_ROOTS),
    "--hsmm-order", str(HSMM_ORDER),
    "--compare-ground-truth" if COMPARE_GROUND_TRUTH else "--no-compare-ground-truth",
]
if PIPELINE == "gaussian":
    cli_args.extend(["--beat-tracker", BEAT_TRACKER])
if DEVICE is not None:
    cli_args.extend(["--device", DEVICE])
for song_name in SONG_NAMES:
    cli_args.extend(["--song", str(song_name)])
for model_name in GAUSSIAN_MODELS:
    cli_args.extend(["--model", str(model_name)])
if AUDIO_DATASET_DIR is not None:
    cli_args.extend(["--audio-dataset", str(AUDIO_DATASET_DIR)])
if ANNOTATION_DATASET_DIR is not None:
    cli_args.extend(["--annotation-dataset", str(ANNOTATION_DATASET_DIR)])

args = build_inference_parser().parse_args(cli_args)
status = run_inference(args)
if status != 0:
    raise RuntimeError(f"Inference failed with exit status {status}.")

summary_path = OUTPUT_DIR / "summary.json"
inference_summary = json.loads(summary_path.read_text(encoding="utf-8"))
print(f"\nInference summary: {summary_path}")

No selected song has ground truth; charts will show predictions only.

Running hierarchical CNN + order-2 HSMM: 10 - The Long And Winding Road (Remastered 2009)
  wrote /Users/hanneswesterberg/Documents/GitHub/dt2470-project/output/notebook/hierarchical_cnn_hsmm/10 - The Long And Winding Road (Remastered 2009)_chords.csv
  wrote /Users/hanneswesterberg/Documents/GitHub/dt2470-project/output/notebook/hierarchical_cnn_hsmm/10 - The Long And Winding Road (Remastered 2009)_chords.json
  open /Users/hanneswesterberg/Documents/GitHub/dt2470-project/output/notebook/10 - The Long And Winding Road (Remastered 2009)_viewer.html to listen with live chords

Running hierarchical CNN + order-2 HSMM: HEY JUDE
  wrote /Users/hanneswesterberg/Documents/GitHub/dt2470-project/output/notebook/hierarchical_cnn_hsmm/HEY JUDE_chords.csv
  wrote /Users/hanneswesterberg/Documents/GitHub/dt2470-project/output/notebook/hierarchical_cnn_hsmm/HEY JUDE_chords.json
  open /Users/hanneswesterberg/Documents/GitHub/dt2

## 3. Inspect predictions and generated files

The application writes timestamped chord CSV files, a JSON summary, and interactive HTML viewers to `output/notebook/`. Gaussian inference also writes comparison charts. This cell lists the generated result files and displays links to the HTML viewers.

In [24]:
print(json.dumps(inference_summary, indent=2))
viewer_paths = sorted(OUTPUT_DIR.glob("*_viewer.html"))
for viewer_path in viewer_paths:
    display(FileLink(viewer_path))

result_csvs = sorted(OUTPUT_DIR.rglob("*_chords.csv"))
print("Prediction CSV files:")
for result_path in result_csvs:
    print(f"  {result_path.relative_to(PROJECT_ROOT)}")

[
  {
    "pipeline": "hierarchical",
    "model": "hierarchical_cnn_hsmm",
    "song": "10 - The Long And Winding Road (Remastered 2009)",
    "audio": "/Users/hanneswesterberg/Documents/GitHub/dt2470-project/input/10 - The Long And Winding Road (Remastered 2009).mp3",
    "prediction_csv": "/Users/hanneswesterberg/Documents/GitHub/dt2470-project/output/notebook/hierarchical_cnn_hsmm/10 - The Long And Winding Road (Remastered 2009)_chords.csv",
    "prediction_json": "/Users/hanneswesterberg/Documents/GitHub/dt2470-project/output/notebook/hierarchical_cnn_hsmm/10 - The Long And Winding Road (Remastered 2009)_chords.json",
    "viewer": "/Users/hanneswesterberg/Documents/GitHub/dt2470-project/output/notebook/10 - The Long And Winding Road (Remastered 2009)_viewer.html",
    "intervals": 106,
    "posterior_cache_hit": true,
    "cnn_checkpoint": "models/hierarchical/best_model.pt",
    "language_model": "models/hierarchical/harmonic_model.pkl",
    "hsmm_order": 2,
    "top_roots": 4
 

/Users/hanneswesterberg/Documents/GitHub/dt2470-project/output/notebook/10 - The Long And Winding Road (Remastered 2009)_viewer.html

/Users/hanneswesterberg/Documents/GitHub/dt2470-project/output/notebook/HEY JUDE_viewer.html

Prediction CSV files:
  output/notebook/hierarchical_cnn_hsmm/10 - The Long And Winding Road (Remastered 2009)_chords.csv
  output/notebook/hierarchical_cnn_hsmm/HEY JUDE_chords.csv


In [25]:
if not result_csvs:
    print("No prediction CSVs found. Run the inference cell first.")
else:
    timeline_path = result_csvs[0]
    with timeline_path.open(encoding="utf-8", newline="") as stream:
        events = list(csv.DictReader(stream))
    if not events:
        raise ValueError(f"Prediction timeline is empty: {timeline_path}")

    starts = np.array([float(event["start_seconds"]) for event in events])
    ends = np.array([float(event["end_seconds"]) for event in events])
    labels = [event["chord"] for event in events]
    figure, axis = plt.subplots(figsize=(14, 2.8), layout="constrained")
    for start, end, label in zip(starts, ends, labels):
        axis.barh(0, end - start, left=start, height=0.55, color="#7aa6c2")
        axis.text((start + end) / 2, 0, label, ha="center", va="center", fontsize=8)
    axis.set(title=timeline_path.stem, xlabel="Time (seconds)", yticks=[])
    axis.set_xlim(0, ends.max())
    plt.show()
    display(FileLink(timeline_path))

/var/folders/nl/l_0_0bts6xn9t76n2fzcy4_80000gn/T/ipykernel_65480/2964915880.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/Users/hanneswesterberg/Documents/GitHub/dt2470-project/output/notebook/hierarchical_cnn_hsmm/10 - The Long And Winding Road (Remastered 2009)_chords.csv

In [26]:
if SOURCE == "local" and local_audio_files:
    selected_audio = [
        path for path in local_audio_files
        if not SONG_NAMES or path.name in SONG_NAMES or path.stem in SONG_NAMES
    ]
    if selected_audio:
        display(Audio(filename=str(selected_audio[0])))
else:
    print("Inline playback is available here for local input/ audio files.")

## 4. Optional model training

Training is disabled by default. The Gaussian baseline trains from paired AAM audio and annotations. The hierarchical CNN trains from precomputed CQT72 feature archives and the frozen split manifest; its feature dataset is not included in this repository. Enable only the training path you intend to run, configure the data paths, and then execute its cell. Training functions are imported from the existing scripts so the notebook follows the project's training implementation.

In [27]:
RUN_GAUSSIAN_TRAINING = False
GAUSSIAN_TRAINING_SONGS = 80
GAUSSIAN_TRAINING_SEED = 42
GAUSSIAN_TRAINING_MODEL = f"gaussian_hmm_{GAUSSIAN_TRAINING_SONGS}"
GAUSSIAN_TRAINING_OVERWRITE = False

if RUN_GAUSSIAN_TRAINING:
    from train_gaussian_baseline import build_parser as build_gaussian_training_parser
    from train_gaussian_baseline import run as run_gaussian_training

    training_args = [
        "--target", "full",
        "--model", GAUSSIAN_TRAINING_MODEL,
        "--training-songs", str(GAUSSIAN_TRAINING_SONGS),
        "--seed", str(GAUSSIAN_TRAINING_SEED),
        "--models-dir", str(MODELS_DIR),
        "--non-interactive",
    ]
    if AUDIO_DATASET_DIR is not None:
        training_args.extend(["--audio-dataset", str(AUDIO_DATASET_DIR)])
    if ANNOTATION_DATASET_DIR is not None:
        training_args.extend(["--annotation-dataset", str(ANNOTATION_DATASET_DIR)])
    if GAUSSIAN_TRAINING_OVERWRITE:
        training_args.append("--overwrite")
    status = run_gaussian_training(
        build_gaussian_training_parser().parse_args(training_args)
    )
    if status != 0:
        raise RuntimeError(f"Gaussian training failed with exit status {status}.")
else:
    print("Gaussian training is disabled. Set RUN_GAUSSIAN_TRAINING = True to enable.")

Gaussian training is disabled. Set RUN_GAUSSIAN_TRAINING = True to enable.


In [28]:
RUN_HIERARCHICAL_TRAINING = False
CQT72_FEATURE_DIR = PROJECT_ROOT / "data" / "features_cqt72_v1"
SPLIT_MANIFEST = PROJECT_ROOT / "models" / "hierarchical" / "split_manifest.json"
REFERENCE_CHECKPOINT = PROJECT_ROOT / "models" / "hierarchical" / "best_model.pt"
HIERARCHICAL_TRAINING_OUTPUT = PROJECT_ROOT / "models" / "training_runs" / "notebook_cnn"
CNN_EPOCHS = 100
CNN_PATIENCE = 15
CNN_BATCH_SIZE = 128

if RUN_HIERARCHICAL_TRAINING:
    from train_hierarchical_cnn import build_parser as build_cnn_training_parser
    from train_hierarchical_cnn import train as train_hierarchical_cnn

    training_args = build_cnn_training_parser().parse_args([
        "--feature-dir", str(CQT72_FEATURE_DIR),
        "--split-manifest", str(SPLIT_MANIFEST),
        "--reference-checkpoint", str(REFERENCE_CHECKPOINT),
        "--output-dir", str(HIERARCHICAL_TRAINING_OUTPUT),
        "--epochs", str(CNN_EPOCHS),
        "--patience", str(CNN_PATIENCE),
        "--batch-size", str(CNN_BATCH_SIZE),
    ])
    trained_checkpoint = train_hierarchical_cnn(training_args)
    print(f"Trained checkpoint: {trained_checkpoint}")
else:
    print("Hierarchical CNN training is disabled. Set RUN_HIERARCHICAL_TRAINING = True to enable.")

Hierarchical CNN training is disabled. Set RUN_HIERARCHICAL_TRAINING = True to enable.


## 5. Optional annotated-dataset evaluation

The following cell evaluates the hierarchical model over paired audio and annotations using the project's established evaluation helpers, then writes per-song CSV/JSON results and an aggregate JSON report. It is disabled by default and requires the full dataset. The evaluator maps detailed model labels to the project's supported major/minor triad labels before scoring; this is an approximation, not a full RQS evaluation.

In [ ]:
RUN_DATASET_EVALUATION = False
EVALUATION_LIMIT = None  # Set a small number to smoke-test on a subset.
EVALUATION_OUTPUT_DIR = PROJECT_ROOT / "output" / "notebook_evaluation"

if RUN_DATASET_EVALUATION:
    from data.load_data import find_songs
    from evaluate_hierarchical import evaluate_song, write_results
    from src.hierarchical import HierarchicalArtifacts, HierarchicalRecognitionPipeline

    paired_songs = list(
        find_songs(AUDIO_DATASET_DIR, ANNOTATION_DATASET_DIR).values()
    )
    if EVALUATION_LIMIT is not None:
        paired_songs = paired_songs[:EVALUATION_LIMIT]
    if not paired_songs:
        raise FileNotFoundError("No paired audio and annotation files were found.")

    artifacts = HierarchicalArtifacts.resolve(
        local_dir=MODELS_DIR / "hierarchical"
    )
    evaluation_pipeline = HierarchicalRecognitionPipeline(
        artifacts,
        device=DEVICE,
        top_roots=TOP_ROOTS,
        hsmm_order=HSMM_ORDER,
        cache_dir=PROJECT_ROOT / ".cache" / "chordrec",
    )
    evaluation_results = []
    for index, song in enumerate(paired_songs, start=1):
        print(f"[{index}/{len(paired_songs)}] {song.song_id}")
        evaluation_results.append(
            evaluate_song(evaluation_pipeline, song, EVALUATION_OUTPUT_DIR)
        )
    write_results(EVALUATION_OUTPUT_DIR, evaluation_results)
    print(f"Evaluation reports written to {EVALUATION_OUTPUT_DIR}")
else:
    print("Dataset evaluation is disabled. Set RUN_DATASET_EVALUATION = True to enable.")

RuntimeError: Could not locate the shared Google Drive project folder. Set AAM_DRIVE_DIR, or GOOGLE_DRIVE_EMAIL and AAM_PROJECT_FOLDER, in .env.

### RWC feature-level CNN benchmark

For the project's fixed RWC test split, this optional workflow reuses [`evaluate_rwc.py`](../evaluate_rwc.py). It scores the CNN emissions against cached frame-level feature targets and writes per-song comparisons. This is distinct from the audio-plus-HSMM annotated-dataset evaluation above.

In [ ]:
RUN_RWC_FEATURE_EVALUATION = True
RWC_FEATURE_DIR = PROJECT_ROOT / "data" / "features_cqt72_v1"
RWC_EVALUATION_OUTPUT = PROJECT_ROOT / "output" / "notebook_rwc_evaluation.json"
RWC_SONGS = None  # None uses the project's fixed test songs; or provide song numbers.

if RUN_RWC_FEATURE_EVALUATION:
    import evaluate_rwc as rwc_evaluation
    from src.hierarchical.hierarchical_cnn import HierarchicalCNNRecognizer

    songs_to_score = rwc_evaluation.TEST_SONGS if RWC_SONGS is None else RWC_SONGS
    missing_features = [
        RWC_FEATURE_DIR / f"RWC_P{song:03d}.npz"
        for song in songs_to_score
        if not (RWC_FEATURE_DIR / f"RWC_P{song:03d}.npz").is_file()
    ]
    if missing_features:
        raise FileNotFoundError(
            "Missing RWC feature file(s): "
            + ", ".join(str(path) for path in missing_features[:5])
        )

    rwc_evaluation.OUTPUT_PATH = RWC_EVALUATION_OUTPUT
    recognizer = HierarchicalCNNRecognizer(
        REFERENCE_CHECKPOINT, device=DEVICE
    )
    rwc_results = [
        rwc_evaluation.evaluate_song(song, RWC_FEATURE_DIR, recognizer)
        for song in songs_to_score
    ]
    rwc_report = {
        "songs": songs_to_score,
        "cnn_macro_average": rwc_evaluation.aggregate(rwc_results, "cnn"),
        "per_song": rwc_results,
    }
    RWC_EVALUATION_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    RWC_EVALUATION_OUTPUT.write_text(json.dumps(rwc_report, indent=2), encoding="utf-8")
    print(json.dumps(rwc_report["cnn_macro_average"], indent=2))
    print(f"RWC report: {RWC_EVALUATION_OUTPUT}")
else:
    print("RWC feature evaluation is disabled. Set RUN_RWC_FEATURE_EVALUATION = True to enable.")

RWC feature evaluation is disabled. Set RUN_RWC_FEATURE_EVALUATION = True to enable.


## 6. Project map and next steps

- `src/hierarchical/`: CQT72 preprocessing, CNN, artifacts, and HSMM decoder.
- `src/features/`, `src/hmm/`, `src/models/`, and `src/training/`: Gaussian-HMM baseline and shared training code.
- `src/evaluation/` and `src/visualization/`: metrics, charts, and interactive viewers.
- `data/`: audio discovery and annotation parsing.
- `tests/`: regression tests for the Python implementation.

This notebook is an interactive frontend, not a replacement for the source package or test suite. For command-line workflows and dataset configuration, see [`README.md`](../README.md).